## 00-Split Pitt Data

In [20]:
# import required packages
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### Define paths and constants

In [21]:
DATA_PATH = "../../../data/"
SPLIT_FOLDER = DATA_PATH + "pitt/"
PITT_PATH_AD = DATA_PATH + "TalkBank downloads/Pitt/Dementia/Cookie/"
PITT_PATH_CN = DATA_PATH + "TalkBank downloads/Pitt/Control/Cookie/"

PARTICIPANT_INFO_PREFIX = "@ID"
AGE_THRESHOLD = 65
ID_INDEX = 0
AGE_INDEX = 2

### Read data and keep earliest interview

In [22]:
def keep_first_interview(file_dict, lst):
    if lst[ID_INDEX] not in file_dict:
        file_dict[lst[ID_INDEX]] = lst
    else:
        if lst[AGE_INDEX] < file_dict[lst[ID_INDEX]][AGE_INDEX]:
            file_dict[lst[ID_INDEX]] = lst


def read_data(path, format, group, personal_info=False):
    files = sorted([item for item in os.listdir(path) if item.endswith(format)])
    file_dict = {}

    for fname in files:
        print("Processing files =", fname)
        with open(path + fname, encoding="utf-8") as f:
            lines = f.readlines()
            if personal_info:
                lst = [
                    line for line in lines if line.startswith(PARTICIPANT_INFO_PREFIX)
                ][ID_INDEX].split("|")
                lst = (
                    [fname.split("-")[ID_INDEX]]
                    + [group]
                    + lst[3:5]
                )
                lst[AGE_INDEX] = int(lst[AGE_INDEX].split(";")[0])
                lst.append(lst[AGE_INDEX] < AGE_THRESHOLD)
                keep_first_interview(file_dict, lst)

    return list(file_dict.values())

In [23]:
pitt_id_train_data = {"ad": [], "cn": []}

In [24]:
format = "cha"
pitt_id_train_data["ad"] = read_data(PITT_PATH_AD, format, "ad", True)
pitt_id_train_data["cn"] = read_data(PITT_PATH_CN, format, "cn", True)

Processing files = 001-0.cha
Processing files = 001-2.cha
Processing files = 003-0.cha
Processing files = 005-0.cha
Processing files = 005-2.cha
Processing files = 007-1.cha
Processing files = 007-3.cha
Processing files = 010-0.cha
Processing files = 010-1.cha
Processing files = 010-2.cha
Processing files = 010-3.cha
Processing files = 010-4.cha
Processing files = 014-2.cha
Processing files = 016-0.cha
Processing files = 016-1.cha
Processing files = 016-3.cha
Processing files = 016-4.cha
Processing files = 018-0.cha
Processing files = 023-0.cha
Processing files = 023-2.cha
Processing files = 024-1.cha
Processing files = 024-2.cha
Processing files = 029-0.cha
Processing files = 029-1.cha
Processing files = 030-0.cha
Processing files = 030-1.cha
Processing files = 033-0.cha
Processing files = 033-1.cha
Processing files = 033-2.cha
Processing files = 033-3.cha
Processing files = 033-4.cha
Processing files = 035-0.cha
Processing files = 035-1.cha
Processing files = 039-0.cha
Processing fil

In [25]:
comb_df = pd.DataFrame(
    pitt_id_train_data["ad"] + pitt_id_train_data["cn"],
    columns=["id", "target", "age_of_first_test", "sex", "less_than_65"],
)

### Split data into train, dev, and test

In [26]:
comb_train_df, comb_test_df = train_test_split(
    comb_df,
    test_size=0.2,
    random_state=42,
    stratify=comb_df[["target", "less_than_65", "sex"]],
)

comb_train_df, comb_dev_df = train_test_split(
    comb_train_df,
    test_size=0.25,
    random_state=42,
    stratify=comb_train_df[["target", "less_than_65", "sex"]],
)

### Generate split CSV files

In [27]:
comb_train_df.to_csv(SPLIT_FOLDER + "pitt_train.csv", index=False)
comb_dev_df.to_csv(SPLIT_FOLDER + "pitt_dev.csv", index=False)
comb_test_df.to_csv(SPLIT_FOLDER + "pitt_test.csv", index=False)